In [0]:
def test_dataframe_not_empty(df, df_name="DataFrame"):
    """Asserts that a DataFrame is not empty."""
    count = df.count()
    assert count > 0, f"❌ TEST FAILED: {df_name} is empty!"
    print(f"✓ TEST PASSED: {df_name} contains {count} records.")
    return count

def test_columns_exist(df, expected_columns, df_name="DataFrame"):
    """Asserts that a DataFrame contains expected columns."""
    actual_columns = set(df.columns)
    missing = set(expected_columns) - actual_columns
    assert len(missing) == 0, f"❌ TEST FAILED: {df_name} is missing columns: {missing}"
    print(f"✓ TEST PASSED: All expected columns present in {df_name}.")

def test_table_exists_and_row_count(spark, catalog, schema, table_name, expected_count):
    """Asserts that a Delta table exists and matches expected row count."""
    full_table = f"{catalog}.{schema}.{table_name}"
    assert spark.catalog.tableExists(full_table), f"❌ TEST FAILED: Table '{full_table}' does not exist!"
    
    actual_count = spark.table(full_table).count()
    assert actual_count == expected_count, (
        f"❌ TEST FAILED: Row count mismatch for '{full_table}'! Expected: {expected_count}, Got: {actual_count}"
    )
    print(f"✓ TEST PASSED: Table '{full_table}' exists with {actual_count} records.")

def test_table_tags(spark, catalog, schema, table_name, expected_tags):
    """Asserts that required governance tags are applied to a table."""
    tags_df = spark.sql(f"""
        SELECT tag_name, tag_value 
        FROM {catalog}.information_schema.table_tags 
        WHERE table_name = '{table_name}'
    """)
    applied_tags = {row["tag_name"]: row["tag_value"] for row in tags_df.collect()}
    
    for tag_key, expected_val in expected_tags.items():
        assert applied_tags.get(tag_key) == expected_val, (
            f"❌ TEST FAILED: Tag '{tag_key}' expected '{expected_val}', got '{applied_tags.get(tag_key)}'"
        )
    print(f"✓ TEST PASSED: Governance tags verified for '{table_name}'.")